In [ ]:
import pandas as pd
import mlflow

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import joblib
import sys
import os
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.features.engineer import create_preprocessor, FeatureEngineer
from src.preprocess.preprocessor import preprocess

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

#### **1. Preparing Data**

In [ ]:
df = pd.read_csv("../data/raw/train.csv")
df = preprocess(df=df)

target = "Churn"

X = df.drop(columns=[target])
y = df[target]

In [ ]:
X.head(2)

In [ ]:
y.head(2)

In [ ]:
# train, test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

#### **2. Model Pipeline**

In [ ]:
## 1. Random Forest Pipeline

# full pipeline
pipeline_random = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", RandomForestClassifier(random_state=42))
])

# hyperparameter tuning space
param_dist_random = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10]
}

## 2. Logistic 

# Pipeline cho Logistic Regression
pipeline_logistic = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# hyperparameter tuning space
param_dist_logistic = {
    "model__C": [0.1, 1, 10],          
    "model__penalty": ["l1", "l2"],      
    "model__solver": ["liblinear"]       
}

## 3. XG Boost

# High-performance classification pipeline using XGBoost
pipeline_xgb = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss' # Standard for binary classification
    ))
])

# Hyperparameter tuning space for XGBoost
param_dist_xgb = {
    "model__n_estimators": [100, 200, 500],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 6, 9],
    "model__subsample": [0.7, 0.8, 0.9],
    "model__colsample_bytree": [0.7, 0.8, 0.9]
}

## 4. Gaussian Naive Bayes 

pipeline_nb = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", GaussianNB())
])

# Hyperparameter tuning space for Naive Bayes
param_dist_nb = {
    "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}


#### **3. Model Tuning and Evaluation**

In [ ]:
# List of pipelines and their corresponding parameter distributions
models_to_tune = [
    ("Random Forest", pipeline_random, param_dist_random),
    ("Logistic Regression", pipeline_logistic, param_dist_logistic),
    ("XGBoost", pipeline_xgb, param_dist_xgb),
    ("Naive Bayes", pipeline_nb, param_dist_nb)
]

# Initialize Experiment
mlflow.set_experiment("Customer_Churn_Analysis_Group04")

comparison_metrics = []
best_models_dict = {}

for name, pipeline, param_dist in models_to_tune:
    # --- Start MLflow Run for each model ---
    with mlflow.start_run(run_name=name):
        print(f"--- Tuning & Evaluating: {name} ---")
        
        # 2. Run Tuning
        search = RandomizedSearchCV(
            pipeline, 
            param_distributions=param_dist, 
            n_iter= 20, 
            scoring="f1", 
            cv=3, 
            n_jobs=-1, 
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        
        # 3. Extract Best Model & Predict
        best_model = search.best_estimator_
        best_models_dict[name] = best_model
        y_pred = best_model.predict(X_test)
        
        # 4. Calculate Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1_bin = f1_score(y_test, y_pred)
        f1_mac = f1_score(y_test, y_pred, average="macro")
        
        # --- Log to MLflow ---
        # Log Hyperparameters
        mlflow.log_params(search.best_params_)
        
        # Log Metrics
        mlflow.log_metrics({
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_binary": f1_bin,
            "f1_macro": f1_mac
        })
        
        # Log Model with an example to define the schema (Signature)
        mlflow.sklearn.log_model(
            sk_model=best_model, 
            artifact_path="model",
            input_example=X_test.iloc[[0]]
        )
        
        # Save locally for comparison table
        metrics = {
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 (Binary)": f1_bin,
            "F1 (Macro)": f1_mac,
            "Best_Params": search.best_params_
        }
        comparison_metrics.append(metrics)
        print(f"Logged {name} to MLflow successfully.\n")

# 6. Display Comparison Table
df_comparison = pd.DataFrame(comparison_metrics).set_index("Model")
print(df_comparison.sort_values(by="F1 (Binary)", ascending=False).drop(columns="Best_Params").to_markdown())

In [ ]:
# Confusion Matrix
# Number of models to plot
n_models = len(best_models_dict)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))

for i, (name, model) in enumerate(best_models_dict.items()):
    # 1. Get predictions
    y_pred = model.predict(X_test)
    
    # 2. Compute confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # 3. Plot using Seaborn
    # format: [[TN, FP], [FN, TP]]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
    axes[i].set_xticklabels(['Stay (0)', 'Churn (1)'])
    axes[i].set_yticklabels(['Stay (0)', 'Churn (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# Identify the winner based on F1 Score
winner_name = df_comparison["F1 (Binary)"].idxmax()
final_best_model = best_models_dict[winner_name]

print(f"The best model is: {winner_name}")

In [ ]:
# Feature Importance

# Take final model
model = final_best_model.named_steps["model"]

# Take Feature name after preprocessing
feature_names = final_best_model.named_steps["preprocessor"].get_feature_names_out()

# Importance Feature
importances = model.feature_importances_

# DataFrame
feat_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

# Plot
plt.figure(figsize=(8,5))
plt.barh(feat_imp["Feature"][:10][::-1], feat_imp["Importance"][:10][::-1])
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.show()

#### **4. Save Model**

In [ ]:
os.makedirs("../models", exist_ok=True)
path = "../models/model.pkl"

# Save model
joblib.dump(final_best_model, path)

# Print absolute path
print("Model saved at:", os.path.abspath(path))